<a target="_blank" href="https://colab.research.google.com/github/LPolyakova/Linear_Algebra_for_CS_students/blob/main/CP/LA_CP_06_Linear_transformations.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# ЛА-Комп'ютерний практикум-06. Лінійні перетворення

In [ ]:
# these imports may be useful

import pandas as pd
from scipy import linalg
import numpy as np
import matplotlib.pyplot as plt
from sympy import *  #module to do symbolic calculations
import matplotlib.animation
import networkx as nx

 #### Деякі інструменти, що стануть у нагоді

In [ ]:
# these tools may be useful

# how to fill triangles (or other polygons) with the color
x1 = np.array([0, 0, 1]) #(x, y) -- coordinates of vertices of the triangle
y1 = np.array([0, 1, 0])

x2 = np.array([1, 1, 2])
y2 = np.array([0, 1, 0])

fig = plt.figure(figsize=(4, 2))
ax = fig.gca()
ax.fill(x1, y1, 'b')
ax.fill(x2, y2, 'b')


In [ ]:
# compute the eigenvalues and eigenvectors of a matrix
A = np.array([[1, 2], [2, 1]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print(f'eigenvalues {eigenvalues}')
print(f'eigenvectors {eigenvectors}') # the columns are eigenvectors !

In [ ]:
# compute characteristic polynomial symbolically with sympy
M = Matrix([[1,0,0], [0,2,0], [0,0,3]])
lamda = symbols('lamda') #змінна, від якої буде многочлен
p = M.charpoly(lamda)
p

## Вправи

💻  **6.1. Із $a$ в $b$.** Нехай $\varphi$ -- лінійне перетворення в $\mathbb{R}^n$, яке має переводити вектори $a_1, a_2, \dots, a_n$ відповідно у вектори $b_1, b_2, \dots, b_n,$ тобто $\varphi(a_j)=b_j$ для $j=1, \dots, n$. Чи завжди таке перетворення існує? Напишіть функцію, яка за заданими координатами векторів $a_j$, $b_j$ в стандартному базисі знаходить матрицю $\varphi$ (також у стандартному базисі), або ж повідомляє, що шуканого перетворення не існує.

# Чи завжди існує лінійне перетворення φ, що переводить a_j у b_j?

# Формулювання задачі

Маємо два набори векторів у просторі ℝⁿ:

- a₁, a₂, …, aₙ — початкові вектори
- b₁, b₂, …, bₙ — їх образи

Потрібно знайти **лінійне перетворення** φ: ℝⁿ → ℝⁿ, яке задовольняє умову:

$$
\varphi(a_j) = b_j, \quad \text{для всіх } j = 1, \dots, n
$$

---

# Коли таке перетворення існує?

Таке лінійне перетворення **існує і єдине**, **якщо і тільки якщо вектори** $a_1, a_2, \dots, a_n$ **утворюють базис простору** $\mathbb{R}^n$.

---

# Коли не існує?

Якщо вектори $a_j$ **лінійно залежні**, тобто **не утворюють базис**, тоді:

- Ми не можемо однозначно задати значення відображення $\varphi$ на всьому просторі.
- Вектори $a_j$ не породжують $\mathbb{R}^n$, отже, не можна побудувати $\varphi$, визначене на всіх векторах простору.
- У такому випадку система для знаходження матриці $\varphi$ є або **неоднозначною**, або **несумісною**.

---

# Висновок

> **Лінійне перетворення** $\varphi$, що переводить $a_j \mapsto b_j$,  
> **існує тоді і тільки тоді**, коли вектори $a_j$ утворюють базис у $\mathbb{R}^n$.  
> У такому разі перетворення **єдине** і його можна знайти з формули:

$$
\varphi = B \cdot A^{-1}
$$

де  
- $A = [a_1\ |\ a_2\ |\ \dots\ |\ a_n]$,  
- $B = [b_1\ |\ b_2\ |\ \dots\ |\ b_n]$.


In [53]:
import numpy as np

def linear_map_matrix(a_vectors: list[list[float]], 
                      b_vectors: list[list[float]]) -> np.ndarray | None:
    """
    Знаходить матрицю лінійного відображення Φ: R^n → R^m, яка переводить вектори a_j → b_j.

    Вектори a_j повинні утворювати базис простору R^n (тобто бути лінійно незалежними
    і їх кількість повинна дорівнювати розмірності простору n).
    Матриця Φ буде розміром m x n.

    Параметри:
    - a_vectors: список векторів a_j (кожен вектор a_j є списком координат).
                 Передбачається, що це вектори-стовпці для матриці A.
                 Наприклад, [[1,0], [0,1]] для a_1=[1,0]^T, a_2=[0,1]^T.
                 Кількість векторів a_j повинна дорівнювати розмірності n.
    - b_vectors: список векторів b_j (кожен вектор b_j є списком координат).
                 Передбачається, що це вектори-стовпці для матриці B.
                 Кількість векторів b_j повинна дорівнювати кількості векторів a_j.
                 Розмірність векторів b_j (m) може відрізнятися від n.

    Повертає:
    - Матрицю Φ (numpy.ndarray розміром m x n), або None, якщо вектори a_j не утворюють базис R^n.
    """
    try:
        # Створюємо матриці A та B, де стовпці - це вектори a_j та b_j відповідно.
        # np.array(vectors).T перетворює список [[v1_x, v1_y], [v2_x, v2_y]]
        # у матрицю [[v1_x, v2_x], [v1_y, v2_y]]
        A = np.array(a_vectors, dtype=float).T
        B = np.array(b_vectors, dtype=float).T
    except ValueError as e:
        print(f"Помилка при створенні матриць з векторів: {e}")
        print("Переконайтеся, що всі вектори в a_vectors мають однакову розмірність n,")
        print("і всі вектори в b_vectors мають однакову розмірність m.")
        return None

    # Перевірка розмірностей
    if A.shape[0] != A.shape[1]: # n x n
        print(f"Помилка: Кількість векторів a_j ({A.shape[1]}) повинна дорівнювати їх розмірності ({A.shape[0]}) для утворення базису R^n.")
        return None
    
    if A.shape[1] != B.shape[1]: # кількість векторів a_j та b_j
        print(f"Помилка: Кількість векторів a_j ({A.shape[1]}) не співпадає з кількістю векторів b_j ({B.shape[1]}).")
        return None

    # n - розмірність простору R^n (і кількість векторів a_j)
    # m - розмірність простору R^m (розмірність векторів b_j)
    n = A.shape[0]
    # m = B.shape[0] # not strictly needed for rank check of A

    # Перевірка: чи A є невиродженою (чи є a_j базисом R^n)
    if np.linalg.matrix_rank(A) < n:
        print(f"Перетворення не може бути однозначно визначене: вектори a_j не утворюють базис простору R^{n}.")
        print(f"Ранг матриці A ({np.linalg.matrix_rank(A)}) менший за розмірність ({n}).")
        return None
    
    # Знаходимо матрицю Φ, що виконує Φ * A = B.
    # Це еквівалентно розв'язанню системи лінійних рівнянь A^T * Φ^T = B^T для Φ^T.
    # Потім Φ = (Φ^T)^T.
    # Або Φ = B * A⁻¹
    try:
        # phi_matrix_T = np.linalg.solve(A.T, B.T)
        # phi_matrix = phi_matrix_T.T
        # Або більш прямо:
        A_inv = np.linalg.inv(A)
        phi_matrix = B @ A_inv
    except np.linalg.LinAlgError:
        # Цей випадок теоретично не мав би виникати, якщо rank_check пройшов,
        # але про всяк випадок.
        print("Помилка обчислення: не вдалося знайти обернену матрицю або розв'язати систему.")
        return None

    print("Матриця лінійного відображення Φ (у стандартному базисі):")
    print(phi_matrix)
    
    return phi_matrix

# Приклад використання:
if __name__ == '__main__':
    print("Приклад 1: R^2 -> R^2 (стандартний базис -> масштабування)")
    a_vectors1 = [[1, 0], [0, 1]]  # a1 = [1,0]^T, a2 = [0,1]^T
    b_vectors1 = [[2, 0], [0, 3]]  # b1 = [2,0]^T, b2 = [0,3]^T (Φa1=b1, Φa2=b2)
    # Очікувана Φ = [[2, 0], [0, 3]]
    phi1 = linear_map_matrix(a_vectors1, b_vectors1)
    if phi1 is not None:
        # Перевірка:
        A1_test = np.array(a_vectors1).T
        B1_test_manual = phi1 @ A1_test
        print("Перевірка: Φ * A:")
        print(B1_test_manual)
        print("Очікувана B:")
        print(np.array(b_vectors1).T)
    print("-" * 30)

    print("Приклад 2: R^2 -> R^2 (інший базис)")
    a_vectors2 = [[1, 1], [1, -1]] # a1 = [1,1]^T, a2 = [1,-1]^T (лінійно незалежні)
    b_vectors2 = [[1, 0], [0, 1]]  # b1 = [1,0]^T, b2 = [0,1]^T
    # A = [[1, 1], [1, -1]], B = [[1, 0], [0, 1]] (тобто I)
    # Φ * A = B  => Φ = B * A⁻¹ = A⁻¹
    # A⁻¹ = 1/(-2) * [[-1, -1], [-1, 1]] = [[0.5, 0.5], [0.5, -0.5]]
    phi2 = linear_map_matrix(a_vectors2, b_vectors2)
    if phi2 is not None:
        A2_test = np.array(a_vectors2).T
        B2_test_manual = phi2 @ A2_test
        print("Перевірка: Φ * A:")
        print(B2_test_manual)
        print("Очікувана B:")
        print(np.array(b_vectors2).T)
    print("-" * 30)

    print("Приклад 3: R^2 -> R^3")
    a_vectors3 = [[1, 0], [0, 1]]
    b_vectors3 = [[1, 2, 3], [4, 5, 6]] # b1 = [1,2,3]^T, b2 = [4,5,6]^T
    # Очікувана Φ = [[1, 4], [2, 5], [3, 6]]
    phi3 = linear_map_matrix(a_vectors3, b_vectors3)
    if phi3 is not None:
        A3_test = np.array(a_vectors3).T
        B3_test_manual = phi3 @ A3_test
        print("Перевірка: Φ * A:")
        print(B3_test_manual) # має бути [[1,4],[2,5],[3,6]] @ [[1,0],[0,1]] = [[1,4],[2,5],[3,6]]
        print("Очікувана B:")
        print(np.array(b_vectors3).T) # [[1,4],[2,5],[3,6]]
    print("-" * 30)
    
    print("Приклад 4: Вектори a_j не є базисом (лінійно залежні)")
    a_vectors4 = [[1, 1], [2, 2]] # a2 = 2*a1
    b_vectors4 = [[1, 0], [0, 1]]
    phi4 = linear_map_matrix(a_vectors4, b_vectors4)
    print("-" * 30)

    print("Приклад 5: Неправильна кількість векторів a_j для R^n (не квадратна A)")
    a_vectors5 = [[1,0]] # 1 вектор у R^2
    b_vectors5 = [[1,2,3]]
    phi5 = linear_map_matrix(a_vectors5, b_vectors5)
    print("-" * 30)

    print("Приклад 6: Неспівпадіння кількості векторів a_j та b_j")
    a_vectors6 = [[1,0], [0,1]]
    b_vectors6 = [[1,2,3]] # лише один b_j
    phi6 = linear_map_matrix(a_vectors6, b_vectors6)

Приклад 1: R^2 -> R^2 (стандартний базис -> масштабування)
Матриця лінійного відображення Φ (у стандартному базисі):
[[2. 0.]
 [0. 3.]]
Перевірка: Φ * A:
[[2. 0.]
 [0. 3.]]
Очікувана B:
[[2 0]
 [0 3]]
------------------------------
Приклад 2: R^2 -> R^2 (інший базис)
Матриця лінійного відображення Φ (у стандартному базисі):
[[ 0.5  0.5]
 [ 0.5 -0.5]]
Перевірка: Φ * A:
[[1. 0.]
 [0. 1.]]
Очікувана B:
[[1 0]
 [0 1]]
------------------------------
Приклад 3: R^2 -> R^3
Матриця лінійного відображення Φ (у стандартному базисі):
[[1. 4.]
 [2. 5.]
 [3. 6.]]
Перевірка: Φ * A:
[[1. 4.]
 [2. 5.]
 [3. 6.]]
Очікувана B:
[[1 4]
 [2 5]
 [3 6]]
------------------------------
Приклад 4: Вектори a_j не є базисом (лінійно залежні)
Перетворення не може бути однозначно визначене: вектори a_j не утворюють базис простору R^2.
Ранг матриці A (1) менший за розмірність (2).
------------------------------
Приклад 5: Неправильна кількість векторів a_j для R^n (не квадратна A)
Помилка: Кількість векторів a_j (1) 

💻  **6.2. Художник.** Намалюйте на площині квадрат із центром у точці $(0,0)$.
Розбийте його на 4 частини (по чотирьох чвертях) та заповніть їх точками чотирьох кольорів.

1. Поекспериментуйте з тим, на що перетвориться квадрат, якщо застосовувати до його точок різні лінійні перетворення: розтягування/стиснення (у тому числі з різними коефіцієнтами за осями), обертання, віддзеркалення відносно деякої осі. Якими матрицями задаються ці перетворення?

2. На що перетвориться квадрат, якщо лінійне перетворення задано а) скалярною матрицею; б) діагональною матрицею; в) верхньотрикутною матрицею?

3. Зробіть висновок: як може змінитися форма квадрата під впливом лінійних перетворень.

In [54]:
#write your code here

💻  **6.3. Просунутий художник.** Створіть нескладне анімоване зображення, в якому відбуватиметься обертання деякого об'єкта (стрілки годинника, сонця або зірок на небосхилі, фігурки на каруселі тощо). Використовуйте матрицю обертання для перетворення координат точок. Для анімації можна використовувати модуль matplotlib.animation.


In [55]:
#write your code here

💻  **6.4. Фрактали.** У цій вправі ми використаємо лінійні перетворення для створення самоподібної структури -- фракталу -- а саме деякої версії трикутника Серпінського, названого так на честь польського математика [Вацлава Серпінського](https://en.wikipedia.org/wiki/Wac%C5%82aw_Sierpi%C5%84ski).

1️⃣ Намалюйте на площині трикутник з вершинами в точках $A(0,0)$, $B(1,0)$, $C(0,1)$. Також створіть $2\times 3 $ numpy-масив $ST$ (Sierpinski Triangle), стовпчики якого, міститимуть координати точок. Варто заповнити внутрішність трикутника кольором -- так вийде наочніше. Те, що ви намалювали -- це нульова (початковва) ітерація трикутника Серпінського.

In [56]:
#write your code here
ST = np.array([[0,0], [1,0], [0,1]]).T  # .T for columns to be points

2️⃣ Створення фракталу відбувається за декілька кроків, на кожному з яких до кожного стовпчика масиву $ST$, тобто до кожної вершини $x$ трикутників, що утворюють фрактал, треба застосувати три перетворення:

$F_a(x) = Ax$, де $A=\left(\begin{array}{cc} 0 & 0.5\\ 0.5 & 0 \end{array}\right)$.

$F_b(x) = Ax + b$ з тією самою $A$ і $b=\left(\begin{array}{c} 0.5 \\ 0  \end{array}\right)$.

$F_c(x)= Ax + c$ з тією самою $A$ і $c=\left(\begin{array}{c} 0 \\ 0.5  \end{array}\right)$.

Поясніть з точки зору геометрії, що робить кожне з цих перетворень.

✔️ *Ваші міркування запишіть тут.*

Зверніть увагу, що застосувати перетворення $F_a$ до всіх вершин трикутника Серпінського одночасно можна, помноживши матрицю $A$ на матрицю $ST$. А для того, щоб застосувати перетворення $F_b$ достатньо помножити $A$ на $ST$ та додати вектор $b$ (він додасться до кожного стовпчика). Аналогічно для $F_c$.

Застосуйте всі перетворення. Від кожного перетворення ви отримаєте по три точки, отже, загалом вийде 3 трикутники на першій ітерації трикутника Серпінського. Подумайте, як заповнити їх кольором. Має вийти щось схоже на наступний малюнок

<img src='images\Sierpinski_triangle_ortho_1.png' width=240, heigth=240>

Також змініть масив $ST$ -- тепер він має містити 9 точок, деякі з яких співпадають.

In [57]:
#write your code here

3️⃣ Продовжіть і зробіть ще кілька ітерацій трикутника Серпінського, щокроку, перетвороюючи кожен із старих трикутників на три нових, вдвічі менших. Подумайте, як заповнювати трикутники кольором. Наприклад, на другій ітерації має вийти щось схоже на наступний малюнок, а новий масив міститиме 27 точок.

<img src='images\Sierpinski_triangle_ortho.png' width=240, heigth=240>


In [58]:
#write your code here

4️ Створіть функцію, що обчислює точки трикутника Серпінського на заданій ітерації, а також функцію, що малює цей трикутник.

In [59]:
#write your code here

💻  **6.5. Характеристичний многочлен.** Характеристичний многочлен квадратної матриці $A$ задається як $\chi(\lambda) =
\det(A-\lambda I)$. Його корені є власними значеннями матриці  $A$. Поекспериментуйте зі знаходженням  $\chi(\lambda)$ для деяких матриць  $A$.

1.  Знайдіть характеристичний многочлен матриці за допомогою символьних обчислень (charpoly в пакеті sympy). Навчіться отримувати коефіцієнти цього многочлена.
2.  Знайдіть власні значення за допомогою чисельних методів (numpy.linalg.eig). Будьте уважні: вони можуть виявитися комплексними.
3.  Побудуйте за знайденими власними значеннями многочлен, коренями якого вони є. За допомогою символьних обчислень (модуль  poly з пакету sympy), або знайдіть коєфіцієнти за допомогою формул Вієта.
4. Протестуйте для різних матриць і порівняйте результати з пунктів 1 і 3.
5. Перевірте, чи виконується для знайдених вами многочленів і вихідної матриці теорема Гамільтона-Келі: $\chi(A)=0$ (тобто якщо підставити матрицю до її характеристичного многочлена та обчислити значення, то отримаємо нульову матрицю).
    

In [60]:
#write your code here

## Задачі

In [61]:
# import section

🧩 **6.6. Степеневий метод.** У цій задачі ми розглянемо один з чисельних методів знаходження максимального власного значення й відповідного власного вектора. На відміну від теоретичного метода, що передбачає факторизацію характеристичного многочлена (а отже, потенційно є дуже ресурсоємним), цей метод здійснює нескладні чисельні операції. Зазвичай його використовують для великих розріджених матриць. Наприклад, Google використовує його для ранжування сторінок в інтернеті (див. наступну задачу).

Ідею розглянемо на прикладі матриці $A$ порядку 2. Припустимо, що матриця має два дійсні різні власні значення: $\lambda_1$, $\lambda_2$, причому $|\lambda_1|>|\lambda_2|$, і відповідні власні вектори $v_1, v_2$. Тоді для довільного вектора $x$ ми можемо записати $x=x_1v_1+x_2v_2$, застосувати матрицю $A$ до обох частин й отримати $Ax=x_1\lambda_1v_1+x_2\lambda_2v_2.$

Будемо продовжувати множити на матрицю $A$ обидві частини рівності. На $k$-ому кроці отримаємо $A^kx= x_1\lambda_1^kv_1+x_2\lambda_2^kv_2$.

А поділивши обидві частини на $\lambda_1^k$, отримаємо $$\dfrac{A^kx}{\lambda_1^k}= x_1v_1+x_2\left(\dfrac{\lambda_2}{\lambda_1}\right)^kv_2 $$


1️⃣  
1. Поясніть, чому можна подати кожен вектор $x$ у вигляді $x=x_1v_1+x_2v_2$.
2. Поясніть за допомогою мат.індукції рівність для $A^kx$.
3. Зробіть висновок, до чого прямує права частина рівності для $\dfrac{A^kx}{\lambda_1^k}$ зі зростанням $k$ і чому.
4. Якщо матриця буде більшого порядку (тобто матиме більшу кількість власних значень), чи зможемо ми зробити такий самий висновок? За яких умов?


Припущення:
*   Матриця $A$ має порядок 2 (тобто діє на 2-вимірному просторі, наприклад $\mathbb{R}^2$).
*   Матриця має два дійсні *різні* власні значення $\lambda_1, \lambda_2$.
*   $v_1, v_2$ – відповідні власні вектори.

Фундаментальна властивість власних векторів полягає в тому, що власні вектори, які відповідають *різним* власним значенням, є лінійно незалежними.
Оскільки $\lambda_1 \neq \lambda_2$, то власні вектори $v_1$ та $v_2$ є лінійно незалежними.
У 2-вимірному просторі будь-які два лінійно незалежні вектори утворюють базис цього простору. Це означає, що будь-який вектор $x$ у цьому 2-вимірному просторі може бути однозначно поданий як лінійна комбінація базисних векторів $v_1$ та $v_2$.
Таким чином, існують унікальні скаляри $x_1$ та $x_2$ такі, що $x = x_1v_1 + x_2v_2$.

**2. Поясніть за допомогою мат.індукції рівність для $A^kx = x_1\lambda_1^kv_1+x_2\lambda_2^kv_2$.**

**База індукції (k=1):**
Маємо $x = x_1v_1 + x_2v_2$.
Помножимо на $A$:
$Ax = A(x_1v_1 + x_2v_2)$
Завдяки лінійності множення матриці на вектор:
$Ax = x_1(Av_1) + x_2(Av_2)$
За визначенням власних векторів і власних значень, $Av_1 = \lambda_1v_1$ та $Av_2 = \lambda_2v_2$.
Підставляючи це, отримуємо:
$A^1x = x_1\lambda_1^1v_1 + x_2\lambda_2^1v_2$.
База індукції вірна.

**Індукційне припущення (для k=m):**
Припустимо, що рівність вірна для деякого натурального числа $m \ge 1$:
$A^mx = x_1\lambda_1^mv_1 + x_2\lambda_2^mv_2$.

**Індукційний крок (доведення для k=m+1):**
Розглянемо $A^{m+1}x$:
$A^{m+1}x = A(A^mx)$
Використовуючи індукційне припущення, підставимо вираз для $A^mx$:
$A^{m+1}x = A(x_1\lambda_1^mv_1 + x_2\lambda_2^mv_2)$
Завдяки лінійності:
$A^{m+1}x = x_1\lambda_1^m(Av_1) + x_2\lambda_2^m(Av_2)$
Знову використовуючи визначення власних векторів ($Av_1 = \lambda_1v_1$, $Av_2 = \lambda_2v_2$):
$A^{m+1}x = x_1\lambda_1^m(\lambda_1v_1) + x_2\lambda_2^m(\lambda_2v_2)$
$A^{m+1}x = x_1\lambda_1^{m+1}v_1 + x_2\lambda_2^{m+1}v_2$.
Цей вираз відповідає формулі для $k=m+1$.
 За принципом математичної індукції, рівність $A^kx = x_1\lambda_1^kv_1 + x_2\lambda_2^kv_2$ вірна для всіх натуральних $k \ge 1$.

**3. Зробіть висновок, до чого прямує права частина рівності для $\dfrac{A^kx}{\lambda_1^k}$ зі зростанням $k$ і чому.**

Рівність: $\dfrac{A^kx}{\lambda_1^k} = x_1v_1 + x_2\left(\dfrac{\lambda_2}{\lambda_1}\right)^kv_2$.

Нам дано, що $|\lambda_1| > |\lambda_2|$.
Розглянемо член $\left(\dfrac{\lambda_2}{\lambda_1}\right)^k$.
Оскільки $|\lambda_1| > |\lambda_2|$, то $\left|\dfrac{\lambda_2}{\lambda_1}\right| < 1$.
Нехай $r = \dfrac{\lambda_2}{\lambda_1}$. Тоді $|r| < 1$.
Коли $k \to \infty$, то $r^k \to 0$. Це стандартна властивість геометричної прогресії зі знаменником, модуль якого менший за одиницю.
Отже, зі зростанням $k$:
$x_2\left(\dfrac{\lambda_2}{\lambda_1}\right)^kv_2 \to x_2 \cdot 0 \cdot v_2 = 0$ (нульовий вектор).

Таким чином, права частина рівності прямує до:
$\lim_{k \to \infty} \left( x_1v_1 + x_2\left(\dfrac{\lambda_2}{\lambda_1}\right)^kv_2 \right) = x_1v_1 + 0 = x_1v_1$.

Права частина рівності $\dfrac{A^kx}{\lambda_1^k}$ прямує до $x_1v_1$ зі зростанням $k$. Це відбувається тому, що член, який містить відношення меншого власного значення до більшого ($\lambda_2/\lambda_1$), піднесений до степеня $k$, прямує до нуля.
Важливою умовою для того, щоб цей метод був корисним для знаходження $v_1$, є те, що $x_1 \neq 0$, тобто початковий вектор $x$ повинен мати ненульову компоненту вздовж напрямку $v_1$.

**4. Якщо матриця буде більшого порядку (тобто матиме більшу кількість власних значень), чи зможемо ми зробити такий самий висновок? За яких умов?**

Так, ми можемо зробити подібний висновок, але за певних умов.

Припустимо, матриця $A$ має порядок $n$ і має $n$ лінійно незалежних власних векторів $v_1, v_2, \dots, v_n$, що відповідають власним значенням $\lambda_1, \lambda_2, \dots, \lambda_n$.
Тоді будь-який вектор $x$ можна подати як $x = \sum_{i=1}^n x_i v_i$.
Аналогічно до випадку 2x2, ми отримаємо:
$A^kx = \sum_{i=1}^n x_i \lambda_i^k v_i$.
Поділимо на $\lambda_1^k$ (припускаючи $\lambda_1 \neq 0$):
$\dfrac{A^kx}{\lambda_1^k} = x_1v_1 + x_2\left(\dfrac{\lambda_2}{\lambda_1}\right)^k v_2 + \dots + x_n\left(\dfrac{\lambda_n}{\lambda_1}\right)^k v_n$.

Щоб цей вираз прямував до $x_1v_1$ зі зростанням $k$, необхідні такі умови:

1.  **Існування базису з власних векторів:** Матриця $A$ повинна бути діагоналізовною, тобто мати $n$ лінійно незалежних власних векторів. Це гарантовано, наприклад, якщо всі $n$ власні значення різні, або якщо матриця симетрична.
2.  **Домінантне власне значення:** Повинно існувати одне власне значення, $\lambda_1$, яке є *строго домінантним* за модулем, тобто $|\lambda_1| > |\lambda_i|$ для всіх $i = 2, 3, \dots, n$. Якщо є кілька власних значень з однаковим максимальним модулем (наприклад, $\lambda_1$ і $-\lambda_1$, або комплексні спряжені), поведінка буде складнішою.
3.  **Ненульова компонента вздовж домінантного власного вектора:** Коефіцієнт $x_1$ (компонента початкового вектора $x$ вздовж $v_1$) повинен бути ненульовим ($x_1 \neq 0$). Якщо $x_1 = 0$, то метод буде збігатися до власного вектора, що відповідає наступному за величиною власному значенню (якщо воно є строго домінантним серед решти).

За цих умов всі доданки $\left(\dfrac{\lambda_i}{\lambda_1}\right)^k$ для $i \ge 2$ прямуватимуть до нуля, оскільки $\left|\dfrac{\lambda_i}{\lambda_1}\right| < 1$, і тоді:
$\lim_{k \to \infty} \dfrac{A^kx}{\lambda_1^k} = x_1v_1$.

Отже якщо матриця діагоналізовна, має строго домінантне за модулем власне значення, і початковий вектор має ненульову проекцію на відповідний домінантний власний вектор.

Отже, зі зростанням $k$  вектор $A^kx$ стає все ближчим до власного вектора, пропорційного $v_1$. Це означає, що ми можемо зробити певну кількість ітерацій множення матриці $A$ на довільний (випадковий) вектор $x$ й отримати наближення власного вектора, що відповідає максимальному власному значенню. Щоправда на кожному кроці модуль наближеного власного вектора може збільшуватися (якщо $\lambda_1>1$) або зменшуватися (якщо $\lambda_1<1$), призводячи до незручних значень координат власного вектора (занадто великих, або занадто малих). Для того, щоб позбутися таких ефектів, ітераційний процес виконують з одночасним нормуванням: $$x_{k+1}=\dfrac{Ax_k}{\|Ax_k\|}.$$

Також з того, що $A^{k}x\approx \lambda_1^{k}x_1v_1$, а $A^{k+1}x\approx \lambda_1^{k+1}x_1v_1$, випливає, що $\lambda_1$  приблизно дорівнює коефіцієнту пропорційності між сусідніми ітераціями: $x_{k+1} \approx \lambda_1x_k $, або ж $\lambda_1 \approx \dfrac{(x_k^t)Ax_k}{(x_k^t)x_k}$.  

2️⃣ Напишіть функцію, яка для дійсної квадратної матриці $A$ за допомогою степеневого методу шукає (нормований) власний вектор, що відповідає найбільшому за модулем власному значенню, а також це власне значення (будьте уважні -- власне значення може виявитися комплексним). Передбачте зупинку ітераційного процесу або по виконанні певної кількості операцій, або якщо різниця модулів векторів, знайдених на сусідніх ітераціях, стає меншою деякого порогового значення. Як початкове наближення, зазвичай, обирають вектор $x$, усі координати якого є однаковими.
Протестуйте функцію. Порівняйте із вбудованими методами пошуку власних значень та векторів. Чи можете ви вказати випадки, коли степеневий метод не буде працювати?

2️⃣ Обґрунтування рівності для $ A^k x $

Нехай початковий вектор $x_0 $ можна подати як лінійну комбінацію власних векторів матриці $ A $:

$$
x_0 = c_1 v_1 + c_2 v_2 + \ldots + c_n v_n
$$

де $ v_i $ — власний вектор, що відповідає власному значенню $ \lambda_i $.

На першій ітерації маємо:

$$
x_1 = A x_0 = A (c_1 v_1 + c_2 v_2 + \ldots + c_n v_n)
$$

$$
= c_1 A v_1 + c_2 A v_2 + \ldots + c_n A v_n
$$

$$
= c_1 \lambda_1 v_1 + c_2 \lambda_2 v_2 + \ldots + c_n \lambda_n v_n
$$

На $ k $-ій ітерації:

$$
x_k = A^k x_0 = c_1 \lambda_1^k v_1 + c_2 \lambda_2^k v_2 + \ldots + c_n \lambda_n^k v_n
$$

Таким чином, загальна формула для ітераційного процесу має вигляд:

$$
x_k = \sum_{i=1}^n c_i \lambda_i^k v_i
$$

In [ ]:
#write your code here
# Power Method Implementation
import numpy as np

def power_method(A, max_iter=1000, tol=1e-6):
    n = A.shape[0]
    # Початкове наближення: вектор з одиниць
    x = np.ones(n)
    x = x / np.linalg.norm(x)

    lambda_old = 0

    for _ in range(max_iter):
        # Множимо матрицю на поточний вектор
        x_new = A @ x
        # Нормалізуємо вектор
        x_new = x_new / np.linalg.norm(x_new)

        # Обчислюємо нове власне значення
        lambda_new = x_new.T @ A @ x_new

        # Перевірка на збіжність
        if np.abs(lambda_new - lambda_old) < tol:
            break

        x = x_new
        lambda_old = lambda_new

    return lambda_new, x

# Тестова матриця
A = np.array([[4, 1], [2, 3]])

# Застосування степеневого методу
lambda_max, eigenvector = power_method(A)

print('Максимальне власне значення (Степеневий метод):', lambda_max)
print('Відповідний власний вектор (Степеневий метод):', eigenvector)

# Перевірка за допомогою вбудованих методів
eigenvalues, eigenvectors = np.linalg.eig(A)
print('Власні значення (np.linalg.eig):', eigenvalues)
print('Власні вектори (np.linalg.eig):')
print(eigenvectors)


3️⃣ Як можна адаптувати цей метод для пошуку найменшого (за модулем) власного значення й відповідного власного вектора?

Адаптація степеневого методу для пошуку найменшого за модулем власного значення

Щоб знайти найменше за модулем власне значення, використовуємо обернену матрицю $ A^{-1} $.  

1. Ітераційний процес для $ A^{-1} $:

$$
x_{k+1} = \frac{A^{-1} x_k}{\| A^{-1} x_k \|}
$$

2. Власні значення матриці $ A^{-1} $ є оберненими до власних значень матриці $ A $:

$$
\mu_i = \frac{1}{\lambda_i}
$$

3. Найбільше власне значення для $ A^{-1} $ відповідає найменшому власному значенню для $ A $:

$$
\lambda_{\min} = \frac{1}{\mu_1}
$$

**Застереження:**  
- Обчислення $ A^{-1} $ є обчислювально складним та може бути нестабільним.  
- Замість явного обчислення оберненої матриці, можна розв'язувати систему $ A y = x $.

4️ Якщо для випадкового вектора $x$, що ви обираєте його на початковому кроці, виявиться що він не містить компоненту, що відповідає власному вектору $v_1$ (тобто якщо $x_1=0$), то яким буде результат ітераційного процесу степеневого методу?

Вплив відсутності компоненти $v_1$ у початковому векторі

Нехай початковий вектор не містить компоненти, що відповідає найбільшому за модулем власному вектору, тобто:

$$
x_0 = c_2 v_2 + c_3 v_3 + \ldots + c_n v_n
$$

Тоді на $ k $-тій ітерації:
$$
x_k = c_2 \lambda_2^k v_2 + c_3 \lambda_3^k v_3 + \ldots + c_n \lambda_n^k v_n
$$

Оскільки $ c_1 = 0$, то зростання $ k $ призведе до того, що домінуватиме компонент із найбільшим за модулем власним значенням серед залишкових.

**Висновок:**
- Якщо $ \lambda_2 $ є унікальним другим за величиною власним значенням, то процес збіжиться до $ v_2 $.
- Якщо кілька власних значень мають однаковий модуль, метод може не збігатися або дати нестабільний результат.
- Отже, початковий вектор має бути обраний так, щоб його проєкція на $ v_1 $ була ненульовою.
